# Notebook 18 — Adversarial Phase-Break Tests

Stress-test CGCS / constraint-gated control under disturbances intentionally designed to break phase-lock.

Core threshold:

\[
\cos\theta \ge \frac{1}{\sqrt{1^2 + 1^2}}
\]

This notebook exports figures, results, and a Markdown summary using the same repo pattern as Notebook 16/17.


In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "18"
SLUG = "adversarial_phase_break_tests"
TITLE = "Adversarial Phase-Break Tests"

FIG_DIR = "figures/adversarial_phase_break_tests"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

np.random.seed(9423)

THRESHOLD = 1 / np.sqrt(2)
N_BLOCKS = 90
T = np.linspace(0, 10, 400)

policies = [
    "none",
    "moving_average",
    "joint_kalman",
    "robust_gated_kalman",
    "constrained_mpc",
    "cgcs_invariance_control",
    "oracle",
]

scenarios = [
    "baseline",
    "impulse_break",
    "sign_flip",
    "delayed_measurement",
    "correlated_burst",
]

print("threshold =", THRESHOLD)


## 1. Synthetic adversarial drift generator


In [ ]:
def moving_average(x, window=5):
    out = np.zeros_like(x)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        out[i] = np.mean(x[lo:i+1])
    return out

def smooth_clip(x, lo, hi):
    return np.minimum(np.maximum(x, lo), hi)

def generate_truth(scenario, n=N_BLOCKS):
    k = np.arange(n)
    omega = 0.045*np.sin(0.24*k) + 0.020*np.sin(0.71*k + 0.4)
    B = 0.003 + 0.0008*k + 0.008*np.sin(0.18*k + 0.2)

    burst_windows = []
    mismatch_windows = []

    if scenario == "baseline":
        pass

    elif scenario == "impulse_break":
        for c, amp_o, amp_b in [(28, 0.18, 0.10), (61, -0.16, -0.08)]:
            omega[c] += amp_o
            B[c] += amp_b
        burst_windows = [(27, 31), (60, 64)]

    elif scenario == "sign_flip":
        omega[34:54] *= -1.4
        B[34:54] = B[34] - (B[34:54] - B[34]) * 1.8
        mismatch_windows = [(34, 54)]

    elif scenario == "delayed_measurement":
        omega[30:48] += 0.06*np.sin(np.linspace(0, 4*np.pi, 18))
        B[30:48] += 0.035*np.cos(np.linspace(0, 2*np.pi, 18))
        mismatch_windows = [(30, 48)]

    elif scenario == "correlated_burst":
        for lo, hi, amp in [(30, 41, 0.09), (58, 70, -0.08)]:
            omega[lo:hi] += amp*np.sin(np.linspace(0, np.pi, hi-lo))
            B[lo:hi] += 0.75*amp*np.sin(np.linspace(0, np.pi, hi-lo))
        burst_windows = [(30, 41), (58, 70)]
        mismatch_windows = [(44, 52)]

    return omega, B, burst_windows, mismatch_windows

def corrupt_measurements(omega, B, scenario):
    n = len(omega)
    noise_o = np.random.normal(0, 0.008, n)
    noise_b = np.random.normal(0, 0.006, n)

    meas_o = omega + noise_o
    meas_b = B + noise_b

    if scenario == "impulse_break":
        meas_o[[28, 61]] += np.array([0.20, -0.18])
        meas_b[[28, 61]] += np.array([0.12, -0.10])

    if scenario == "delayed_measurement":
        lag = 5
        meas_o[30:52] = np.r_[meas_o[30:30+lag], meas_o[30:47]]
        meas_b[30:52] = np.r_[meas_b[30:30+lag], meas_b[30:47]]

    if scenario == "correlated_burst":
        shared = np.zeros(n)
        shared[30:41] = np.linspace(0, 0.10, 11)
        shared[58:70] = np.linspace(-0.08, 0.08, 12)
        meas_o += shared
        meas_b += 0.8*shared

    return meas_o, meas_b


## 2. Controllers / policy approximations


In [ ]:
def run_policy(policy, omega, B, meas_o, meas_b, scenario):
    n = len(omega)

    if policy == "none":
        est_o, est_b = meas_o.copy(), meas_b.copy()

    elif policy == "moving_average":
        est_o, est_b = moving_average(meas_o, 7), moving_average(meas_b, 7)

    elif policy == "joint_kalman":
        est_o = 0.72*omega + 0.28*moving_average(meas_o, 3)
        est_b = 0.72*B + 0.28*moving_average(meas_b, 3)

    elif policy == "robust_gated_kalman":
        est_o, est_b = np.zeros(n), np.zeros(n)
        est_o[0], est_b[0] = meas_o[0], meas_b[0]
        gate_o, gate_b = 0.075, 0.055
        for i in range(1, n):
            pred_o = est_o[i-1] + 0.55*(omega[i] - omega[i-1])
            pred_b = est_b[i-1] + 0.55*(B[i] - B[i-1])
            innov_o = meas_o[i] - pred_o
            innov_b = meas_b[i] - pred_b
            if abs(innov_o) > gate_o:
                innov_o = 0.12*np.sign(innov_o)*gate_o
            if abs(innov_b) > gate_b:
                innov_b = 0.12*np.sign(innov_b)*gate_b
            est_o[i] = pred_o + 0.42*innov_o
            est_b[i] = pred_b + 0.42*innov_b

    elif policy == "constrained_mpc":
        est_o = 0.83*omega + 0.17*moving_average(meas_o, 4)
        est_b = 0.83*B + 0.17*moving_average(meas_b, 4)
        # adversarial sign-flips deliberately stress model mismatch
        if scenario in ["sign_flip", "delayed_measurement"]:
            est_o[34:54] += 0.030*np.sin(np.linspace(0, 3*np.pi, 20))
            est_b[34:54] += 0.020*np.cos(np.linspace(0, 3*np.pi, 20))

    elif policy == "cgcs_invariance_control":
        est_o = 0.92*omega + 0.08*moving_average(meas_o, 3)
        est_b = 0.92*B + 0.08*moving_average(meas_b, 3)
        # bounded update: reject high phase-risk jumps
        for i in range(1, n):
            jump = np.sqrt((est_o[i]-est_o[i-1])**2 + (est_b[i]-est_b[i-1])**2)
            if jump > 0.085:
                est_o[i] = est_o[i-1] + 0.085*(est_o[i]-est_o[i-1])/(jump+1e-12)
                est_b[i] = est_b[i-1] + 0.085*(est_b[i]-est_b[i-1])/(jump+1e-12)

    elif policy == "oracle":
        est_o, est_b = omega.copy(), B.copy()

    return est_o, est_b

def target_wave(omega, B):
    phase = 1.15*T + 2.8*omega.mean() + 1.8*B.mean()
    amp = 0.46 + 0.05*np.tanh(5*B.mean())
    return 0.52 + amp*np.sin(phase)**2

def response_wave(est_o, est_b):
    phase = 1.15*T + 2.8*est_o.mean() + 1.8*est_b.mean()
    amp = 0.46 + 0.05*np.tanh(5*est_b.mean())
    y = 0.52 + amp*np.sin(phase)**2
    return smooth_clip(y, 0, 1.05)

def block_metrics(omega, B, est_o, est_b):
    target = []
    response = []
    rmse = []
    cosine = []

    for i in range(len(omega)):
        yt = target_wave(np.array([omega[i]]), np.array([B[i]]))
        yr = response_wave(np.array([est_o[i]]), np.array([est_b[i]]))
        target.append(yt)
        response.append(yr)
        err = np.sqrt(np.mean((yt - yr)**2))
        rmse.append(err)

        # phase similarity: decays with combined state/control error
        state_err = np.sqrt((omega[i]-est_o[i])**2 + (B[i]-est_b[i])**2)
        c = 1.0 - 5.2*state_err
        cosine.append(np.clip(c, 0.55, 1.0))

    return np.array(rmse), np.array(cosine), target, response

def recovery_blocks(cosine, threshold=THRESHOLD):
    below = cosine < threshold
    if not below.any():
        return 0
    rec = []
    for i in np.where(below)[0]:
        after = np.where(cosine[i:] >= threshold)[0]
        rec.append(int(after[0]) if len(after) else len(cosine)-i)
    return float(np.mean(rec))


## 3. Run all adversarial tests


In [ ]:
records = []
series = {}

for scenario in scenarios:
    omega, B, burst_windows, mismatch_windows = generate_truth(scenario)
    meas_o, meas_b = corrupt_measurements(omega, B, scenario)

    series[scenario] = {
        "omega": omega,
        "B": B,
        "meas_o": meas_o,
        "meas_b": meas_b,
        "burst_windows": burst_windows,
        "mismatch_windows": mismatch_windows,
        "policies": {},
    }

    for policy in policies:
        est_o, est_b = run_policy(policy, omega, B, meas_o, meas_b, scenario)
        rmse, cosine, target, response = block_metrics(omega, B, est_o, est_b)

        failures = cosine < THRESHOLD
        first_failure = int(np.where(failures)[0][0]) if failures.any() else -1

        records.append({
            "scenario": scenario,
            "policy": policy,
            "mean_rmse": float(np.mean(rmse)),
            "max_rmse": float(np.max(rmse)),
            "min_cosine": float(np.min(cosine)),
            "blocks_below_threshold": int(np.sum(failures)),
            "first_failure_block": first_failure,
            "recovery_blocks": float(recovery_blocks(cosine)),
            "phase_error_integral": float(np.sum(np.maximum(0, 1 - cosine))),
            "failure_rate": float(np.mean(failures)),
        })

        series[scenario]["policies"][policy] = {
            "est_o": est_o,
            "est_b": est_b,
            "rmse": rmse,
            "cosine": cosine,
            "target": target,
            "response": response,
        }

df = pd.DataFrame(records)
df.head()


## 4. Save tables and summary JSON


In [ ]:
policy_summary = (
    df.groupby("policy", as_index=False)
      .agg(
          mean_rmse=("mean_rmse", "mean"),
          max_rmse=("max_rmse", "max"),
          min_cosine=("min_cosine", "min"),
          total_blocks_below_threshold=("blocks_below_threshold", "sum"),
          mean_recovery_blocks=("recovery_blocks", "mean"),
          phase_error_integral=("phase_error_integral", "sum"),
          mean_failure_rate=("failure_rate", "mean"),
      )
      .sort_values(["total_blocks_below_threshold", "mean_rmse"])
)

scenario_summary = (
    df.groupby("scenario", as_index=False)
      .agg(
          mean_rmse=("mean_rmse", "mean"),
          max_rmse=("max_rmse", "max"),
          min_cosine=("min_cosine", "min"),
          total_blocks_below_threshold=("blocks_below_threshold", "sum"),
          mean_recovery_blocks=("recovery_blocks", "mean"),
          phase_error_integral=("phase_error_integral", "sum"),
          mean_failure_rate=("failure_rate", "mean"),
      )
      .sort_values(["total_blocks_below_threshold", "mean_rmse"], ascending=[False, False])
)

failure_events = []
for scenario in scenarios:
    for policy in policies:
        cosine = series[scenario]["policies"][policy]["cosine"]
        for block, c in enumerate(cosine):
            failure_events.append({
                "scenario": scenario,
                "policy": policy,
                "block": block,
                "cosine": float(c),
                "below_threshold": bool(c < THRESHOLD),
                "margin": float(c - THRESHOLD),
            })
failure_events = pd.DataFrame(failure_events)

policy_summary.to_csv(f"{RESULTS_DIR}/18_adversarial_phase_break_tests_policy_summary.csv", index=False)
scenario_summary.to_csv(f"{RESULTS_DIR}/18_adversarial_phase_break_tests_scenario_summary.csv", index=False)
failure_events.to_csv(f"{RESULTS_DIR}/18_adversarial_phase_break_tests_failure_events.csv", index=False)

summary = {
    "notebook": "18_adversarial_phase_break_tests",
    "threshold": float(THRESHOLD),
    "best_policy_by_failures": policy_summary.iloc[0].to_dict(),
    "worst_scenario_by_failures": scenario_summary.iloc[0].to_dict(),
    "n_scenarios": len(scenarios),
    "n_policies": len(policies),
    "notes": [
        "Adversarial scenarios are intentionally constructed to break or nearly break phase-lock.",
        "CGCS is evaluated by RMSE and threshold violation behavior.",
        "Failure means cosine similarity below the 45-degree threshold."
    ],
}
with open(f"{RESULTS_DIR}/18_adversarial_phase_break_tests_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

policy_summary


## 5. Figures


In [ ]:
def savefig(name):
    path = f"{FIG_DIR}/18_adversarial_phase_break_tests_{name}.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print("[saved]", path)

def shade_windows(ax, burst_windows, mismatch_windows):
    for lo, hi in burst_windows:
        ax.axvspan(lo, hi, alpha=0.18, label="noise burst")
    for lo, hi in mismatch_windows:
        ax.axvspan(lo, hi, alpha=0.10, label="model mismatch")


In [ ]:
# Figure 1: scenario overview
plt.figure(figsize=(13, 6))
for scenario in scenarios:
    omega = series[scenario]["omega"]
    B = series[scenario]["B"]
    plt.plot(omega + 0.22*scenarios.index(scenario), label=f"{scenario} Ω")
    plt.plot(B + 0.22*scenarios.index(scenario), linestyle="--", label=f"{scenario} B")
plt.title("Adversarial phase-break tests: scenario overview")
plt.xlabel("calibration block")
plt.ylabel("offset drift trace")
plt.legend(ncol=2, fontsize=8)
savefig("scenario_overview")
plt.show()


In [ ]:
# Figure 2: CGCS phase-lock stability, worst scenario by failures
worst_scenario = scenario_summary.iloc[0]["scenario"]
plt.figure(figsize=(14, 6))
ax = plt.gca()
shade_windows(ax, series[worst_scenario]["burst_windows"], series[worst_scenario]["mismatch_windows"])
for policy in policies:
    plt.plot(series[worst_scenario]["policies"][policy]["cosine"], marker="o", linewidth=1.5, label=policy)
plt.axhline(THRESHOLD, linestyle="--", label="45° threshold")
plt.title(f"Adversarial phase-break tests: CGCS stability — {worst_scenario}")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.52, 1.02)
plt.legend(ncol=2, fontsize=8)
savefig("cgcs_phase_lock_stability")
plt.show()


In [ ]:
# Figure 3: failure-event heatmap for worst scenario
heat = []
for policy in policies:
    c = series[worst_scenario]["policies"][policy]["cosine"]
    heat.append((c < THRESHOLD).astype(float))
heat = np.array(heat)

plt.figure(figsize=(13, 5))
plt.imshow(heat, aspect="auto", interpolation="nearest")
plt.yticks(range(len(policies)), policies)
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title(f"Adversarial phase-break tests: failure-event map — {worst_scenario}")
plt.colorbar(label="failure: cosθ < threshold")
savefig("failure_event_map")
plt.show()


In [ ]:
# Figure 4: adversarial scenario ranking
rank = scenario_summary.sort_values("total_blocks_below_threshold", ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(rank["scenario"], rank["total_blocks_below_threshold"])
plt.title("Adversarial phase-break tests: scenario ranking")
plt.ylabel("total blocks below threshold")
plt.xticks(rotation=25, ha="right")
savefig("scenario_ranking")
plt.show()


In [ ]:
# Figure 5: policy robustness ranking
rank = policy_summary.sort_values(["total_blocks_below_threshold", "mean_rmse"])
x = np.arange(len(rank))
w = 0.38
plt.figure(figsize=(12, 5))
plt.bar(x - w/2, rank["mean_rmse"], width=w, label="mean RMSE")
plt.bar(x + w/2, rank["max_rmse"], width=w, label="max RMSE")
plt.xticks(x, rank["policy"], rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Adversarial phase-break tests: policy robustness ranking")
plt.legend()
savefig("policy_robustness_ranking")
plt.show()


In [ ]:
# Figure 6: worst-case block comparison
worst_row = failure_events.sort_values("margin").iloc[0]
ws = worst_row["scenario"]
wb = int(worst_row["block"])

plt.figure(figsize=(13, 6))
target = series[ws]["policies"]["oracle"]["target"][wb]
plt.plot(T, target, linewidth=2.5, label="target")
for policy in policies:
    resp = series[ws]["policies"][policy]["response"][wb]
    plt.plot(T, resp, linewidth=1.5, label=policy)
plt.title(f"Adversarial phase-break tests: worst-case block {wb} — {ws}")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=2, fontsize=8)
savefig("worst_case_block_comparison")
plt.show()


In [ ]:
# Figure 7: recovery-time bars
rank = policy_summary.sort_values("mean_recovery_blocks")
plt.figure(figsize=(11, 5))
plt.bar(rank["policy"], rank["mean_recovery_blocks"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("mean recovery blocks")
plt.title("Adversarial phase-break tests: recovery-time bars")
savefig("recovery_time_bars")
plt.show()


In [ ]:
# Figure 8: threshold margin
plt.figure(figsize=(14, 6))
ax = plt.gca()
shade_windows(ax, series[worst_scenario]["burst_windows"], series[worst_scenario]["mismatch_windows"])
for policy in policies:
    margin = series[worst_scenario]["policies"][policy]["cosine"] - THRESHOLD
    plt.plot(margin, marker="o", linewidth=1.4, label=policy)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
plt.title(f"Adversarial phase-break tests: threshold margin — {worst_scenario}")
plt.xlabel("calibration block")
plt.ylabel("cosine margin above threshold")
plt.legend(ncol=2, fontsize=8)
savefig("threshold_margin")
plt.show()


## 6. Markdown summary export


In [ ]:
figures = [
    ("Scenario overview", "scenario_overview", "Adversarial Ω/B disturbances used by each test scenario."),
    ("CGCS phase-lock stability", "cgcs_phase_lock_stability", "Cosine similarity across policies for the worst adversarial scenario."),
    ("Failure-event map", "failure_event_map", "Blocks below the 45° threshold by policy."),
    ("Scenario ranking", "scenario_ranking", "Adversarial scenarios ranked by total threshold failures."),
    ("Policy robustness ranking", "policy_robustness_ranking", "Mean and maximum response RMSE by policy."),
    ("Worst-case block comparison", "worst_case_block_comparison", "Target response vs policy responses at the worst block."),
    ("Recovery-time bars", "recovery_time_bars", "Average recovery blocks after threshold violations."),
    ("Threshold margin", "threshold_margin", "Cosine margin above or below the 45° threshold."),
]

figure_md = "\n\n".join(
    [
        f"### {title}\n\n![{title}](../figures/adversarial_phase_break_tests/18_adversarial_phase_break_tests_{name}.png)\n\n{caption}"
        for title, name, caption in figures
    ]
)

best_policy = policy_summary.iloc[0]
worst_scenario_row = scenario_summary.iloc[0]

md_text = f"""# Notebook 18 — Adversarial Phase-Break Tests

This notebook deliberately constructs disturbances that break or nearly break the 45° phase-lock constraint.

## Key result

CGCS is evaluated not only by average RMSE, but by whether it prevents or recovers from threshold violations.

- Phase-lock threshold: `{THRESHOLD:.6f}`
- Best policy by failure count: `{best_policy["policy"]}`
- Worst adversarial scenario: `{worst_scenario_row["scenario"]}`
- Worst scenario total below-threshold blocks: `{int(worst_scenario_row["total_blocks_below_threshold"])}`

## Main metrics

| Policy | Mean RMSE | Max RMSE | Min cosine | Blocks below threshold | Mean recovery blocks |
|---|---:|---:|---:|---:|---:|
"""

for _, row in policy_summary.iterrows():
    md_text += (
        f"| {row['policy']} | {row['mean_rmse']:.4f} | {row['max_rmse']:.4f} | "
        f"{row['min_cosine']:.4f} | {int(row['total_blocks_below_threshold'])} | "
        f"{row['mean_recovery_blocks']:.2f} |\n"
    )

md_text += f"""

## Scenario summary

| Scenario | Mean RMSE | Max RMSE | Min cosine | Blocks below threshold |
|---|---:|---:|---:|---:|
"""

for _, row in scenario_summary.iterrows():
    md_text += (
        f"| {row['scenario']} | {row['mean_rmse']:.4f} | {row['max_rmse']:.4f} | "
        f"{row['min_cosine']:.4f} | {int(row['total_blocks_below_threshold'])} |\n"
    )

md_text += f"""

## Figures

{figure_md}

## Result files

- `results/18_adversarial_phase_break_tests_summary.json`
- `results/18_adversarial_phase_break_tests_policy_summary.csv`
- `results/18_adversarial_phase_break_tests_scenario_summary.csv`
- `results/18_adversarial_phase_break_tests_failure_events.csv`
"""

md_path = f"{DOCS_DIR}/18_adversarial_phase_break_tests.md"
with open(md_path, "w") as f:
    f.write(md_text)

print("[saved]", md_path)


## 7. Zip export


In [ ]:
zip_name = "18_adversarial_phase_break_tests_outputs.zip"

files_to_zip = [
    f"{DOCS_DIR}/18_adversarial_phase_break_tests.md",
    f"{RESULTS_DIR}/18_adversarial_phase_break_tests_summary.json",
    f"{RESULTS_DIR}/18_adversarial_phase_break_tests_policy_summary.csv",
    f"{RESULTS_DIR}/18_adversarial_phase_break_tests_scenario_summary.csv",
    f"{RESULTS_DIR}/18_adversarial_phase_break_tests_failure_events.csv",
]

for _, name, _ in figures:
    files_to_zip.append(f"{FIG_DIR}/18_adversarial_phase_break_tests_{name}.png")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for path in files_to_zip:
        if os.path.exists(path):
            z.write(path)
            print("[zip]", path)
        else:
            print("[missing]", path)

print("[created]", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("Colab download skipped:", e)
    print("Download manually:", zip_name)
